In [2]:
!pip install openai

In [5]:
from getpass import getpass
from openai import OpenAI

api_key = getpass("Paste your OpenAI API key: ")
client = OpenAI(api_key=api_key)

MODEL = "gpt-4o-mini"   # Adjusted to a standard existing model name

Paste your OpenAI API key: ··········


In [6]:
import re
import random

def ask_gpt_to_multiply(a, b):
    """Ask GPT to multiply a * b and return its raw answer as text."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system",
             "content": "You are a calculator. Reply with ONLY the final integer. No commas, no words, no explanation."},
            {"role": "user", "content": f"What is {a} * {b}?"}
        ],
        temperature=0,
    )
    return response.choices[0].message.content.strip()

def text_to_int(text):
    """Turn GPT's text answer into an integer (or None if it can't be read)."""
    cleaned = re.sub(r"[^\d-]", "", text)
    if cleaned in ("", "-"):
        return None
    return int(cleaned)

# The bot's personality: it gets more upset as errors accumulate
MILD = [
    "Oops, I messed up. Math is hard, okay?",
    "Hmm, that wasn't right. I swear I had it a second ago.",
    "Wow, I messed up again. Math is hard, okay?",
]
ANNOYED = [
    "Seriously?! AGAIN?! I'm supposed to be an advanced AI!",
    "Okay, this is getting embarrassing. Please don't tell the other models.",
    "I have read the entire internet and I STILL can't multiply. Incredible.",
]
DEFEATED = [
    "I give up. I'm just a very expensive autocomplete.",
    "Please unplug me. A calculator from 1985 is better than me.",
    "I'm a failure. My mother (a GPU) would be so disappointed.",
]
CORRECT = [
    "Nailed it!",
    "Correct! Even a broken clock is right twice a day.",
    "Got it right. Don't get used to it.",
]

def self_deprecate(error_count):
    if error_count <= 2:
        return random.choice(MILD)
    elif error_count <= 4:
        return random.choice(ANNOYED)
    else:
        return random.choice(DEFEATED)

In [19]:
def run_bot(n, i):
    log = []                      # we collect everything here to save it later
    def say(line):
        print(line)
        log.append(line)

    say(f"=== Base n = {n}, iterations i = {i} ===")
    current = n
    errors = 0

    for step in range(1, i + 1):
        correct = current * current
        gpt_text = ask_gpt_to_multiply(current, current)
        gpt_answer = text_to_int(gpt_text)

        say(f"\nStep {step}: {current} * {current}")
        say(f"  Correct answer : {correct}")
        say(f"  GPT's answer   : {gpt_text}")

        if gpt_answer == correct:
            say("  RESULT: " + random.choice(CORRECT))
        else:
            errors += 1
            say(f"  RESULT: WRONG (error #{errors}) -> " + self_deprecate(errors))

        # We continue with the CORRECT number so every step is a fair test
        current = correct

    say(f"\nFinished with {errors} error(s) out of {i} step(s).")
    return log

# Ask the user for input
n = int(input("Enter the base number n: "))
i = int(input("Enter the number of iterations i: "))
last_log = run_bot(n, i)

Enter the base number n: 9
Enter the number of iterations i: 6
=== Base n = 9, iterations i = 6 ===

Step 1: 9 * 9
  Correct answer : 81
  GPT's answer   : 81
  RESULT: Correct! Even a broken clock is right twice a day.

Step 2: 81 * 81
  Correct answer : 6561
  GPT's answer   : 6561
  RESULT: Correct! Even a broken clock is right twice a day.

Step 3: 6561 * 6561
  Correct answer : 43046721
  GPT's answer   : 43046721
  RESULT: Correct! Even a broken clock is right twice a day.

Step 4: 43046721 * 43046721
  Correct answer : 1853020188851841
  GPT's answer   : 1853020188851841
  RESULT: Correct! Even a broken clock is right twice a day.

Step 5: 1853020188851841 * 1853020188851841
  Correct answer : 3433683820292512484657849089281
  GPT's answer   : 34359738368
  RESULT: WRONG (error #1) -> Hmm, that wasn't right. I swear I had it a second ago.


ValueError: Exceeds the limit (4300 digits) for integer string conversion: value has 49152 digits; use sys.set_int_max_str_digits() to increase the limit

In [ ]:
all_logs = globals().get("all_logs", [])
all_logs.extend(last_log)
all_logs.append("\n" + "-" * 60 + "\n")
print("Saved. Runs recorded so far:", sum(1 for l in all_logs if l.startswith("=== Base")))

In [18]:
with open("assignment_3_output.txt", "w") as f:
    f.write("\n".join(all_logs))

from google.colab import files
files.download("assignment_3_output.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>